# A: Baselines and independent criticism (M1 / M2 / M3)

**Research question (part of RQ1):** Does independent criticism (M3) do better than self-reflection (M2) and direct generation (M1)? When does criticism help, and when does it introduce errors?

This notebook reads one or more runner output directories (`results/runs/<run>/records.jsonl` + `gold_lookup.json`, produced by E's runner). Answers are scored with **E's official** `exact_match` / `token_f1` / `paired_bootstrap_ci`; A does not score anything itself. Errors count as 0 in the denominator, as E does. All heavy lifting lives in `research/A_baselines_analysis.py`, so the numbers here and in `results/analysis/A_<run>/` are identical.

**Status labels:** `main_n100` is a **pilot** (prompt set v1). Its failure patterns motivated prompt set v2, so these 100 questions must not be reported as the frozen evaluation. `dev_*` runs use B's 50 dev questions.

In [ ]:
import sys, json
from pathlib import Path
ROOT = Path.cwd().resolve()
while not (ROOT / 'contracts').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT)); sys.path.insert(0, str(ROOT / 'research'))
import A_baselines_analysis as AA

# Edit this list: run directories to analyse (missing ones are skipped).
RUNS = ['main_n100', 'dev_v1', 'dev_fix']
summaries = {}
for name in RUNS:
    run_dir = ROOT / 'results' / 'runs' / name
    if not (run_dir / 'records.jsonl').exists():
        print(f'skip {name}: no records.jsonl'); continue
    summaries[name] = AA.analyse(run_dir, ROOT / 'results' / 'analysis' / f'A_{name}')
    print(f'{name}: {summaries[name]["n_examples"]} examples, A prompts = {summaries[name]["prompt_versions"]}')

## 1. Headline: EM / F1 / errors / cost per method

In [ ]:
try:
    import pandas as pd
except ImportError:
    pd = None
rows = []
for name, s in summaries.items():
    for m, h in s['headline'].items():
        rows.append({'run': name, 'method': m, **h})
if pd is not None and rows:
    display(pd.DataFrame(rows).set_index(['run', 'method']))
else:
    for r in rows: print(r)

## 2. Paired differences (bootstrap 95% CI, E's implementation)

In [ ]:
for name, s in summaries.items():
    print(f'--- {name}')
    for k, v in s['paired_bootstrap'].items():
        print(f"{k:28s} delta={v['delta_obs']:+.3f}  CI=[{v['ci_low']:+.3f}, {v['ci_high']:+.3f}]  n={v['n']}")

## 3. What revision did to the shared draft

Transitions run from the draft to the final answer. `correct->wrong` means the revision introduced an error. The reasons for `correct->wrong` come from a heuristic triage: *format* means the final answer contains the gold or draft answer plus extra words, and *content* means a different answer was given.

In [ ]:
for name, s in summaries.items():
    print(f'--- {name}')
    for m in ('M2', 'M3'):
        print(f'  {m} transitions:', s['transitions'][m])
        print(f'  {m} correct->wrong reasons:', s['correct_to_wrong_reasons'][m])

## 4. Does the critic know when to criticise?

If the critic is informative, it should raise issues much more often on **wrong** drafts than on **correct** ones. If the two rates are about the same, the critic cannot tell which drafts need fixing.

In [ ]:
for name, s in summaries.items():
    print(f'--- {name}')
    for m, f in s['feedback_behaviour'].items():
        print(f"  {m}: issue rate | draft correct = {f['issue_raised_rate_when_draft_correct']}, "
              f"| draft wrong = {f['issue_raised_rate_when_draft_wrong']}, answer changed (ok) = {f['answer_changed_rate_ok']}")

## 5. Failures (status = error) and per question type

In [ ]:
for name, s in summaries.items():
    print(f'--- {name}')
    print('  errors:', json.dumps(s['errors'], ensure_ascii=False))
    print('  by type:', json.dumps(s['by_question_type']))

## 6. Cases for error analysis (see `results/analysis/A_<run>/cases.md` for the full list)

In [ ]:
for name in summaries:
    p = ROOT / 'results' / 'analysis' / f'A_{name}' / 'cases.md'
    text = p.read_text(encoding='utf-8').splitlines()
    print(f'===== {name} ({len(text)} lines); first 40 lines:')
    print('\n'.join(text[:40]))

## 7. Notes for the report
- Always state the sample size, split, backbone, A prompt set and versions (printed above), and that errors count as 0.
- M2 = 2 calls (self-check + self-revise), M3 = 2 calls (critic + D revision), M1 = 0 extra calls.
- Confound: M2 uses A's own revision prompt, while M3 uses D's shared revision. M2 vs M3 changes both the critic and the reviser.
- `main_n100` is a pilot that informed prompt v2. The frozen evaluation must use unseen questions.
- Null or negative findings, such as revision hurting EM, are valid results.